In [83]:
import requests
from urllib.parse import unquote
import xml.etree.ElementTree as xmltree

url = "https://tw.stock.yahoo.com/rss?category=intl-markets"
response = requests.get(url)

# Create ElementTree from xml_string
element = xmltree.fromstring(response.text)
# print(xmltree.dump(element))
interests = ['title', 'link', 'pubDate']

# Iterate over all items in the XML response
for item in element.findall('./channel/item')[0:5]:
 
  # Iterate over all elements in the item
  for b in item:
    # Check if the element is in the list of interests and it has a value
    if b.tag in interests and b.text:
      # Unquote the text and print it
      print(unquote(b.text))
  # Print an empty line
  print()

專家：2024年別做空美科技股七巨頭
https://tw.stock.yahoo.com/news/漲多不等於必會回檔-專家-2024年別做空七巨頭-050600319.html
Tue, 02 Jan 2024 05:06:00 GMT

伊朗軍艦進紅海 國際油價應聲上漲
https://tw.stock.yahoo.com/news/伊朗軍艦進紅海-國際油價應聲上漲-142054334.html
Tue, 02 Jan 2024 14:20:54 GMT

2024年美股能否續強 就業數據是關鍵
https://tw.stock.yahoo.com/news/2024年美股能否續強-就業數據是關鍵-141724592.html
Tue, 02 Jan 2024 14:17:24 GMT

智冠與大股東網銀國際 再為博奕遊戲摃上
https://tw.stock.yahoo.com/news/智冠與大股東網銀國際-再為博奕遊戲摃上-141520538.html
Tue, 02 Jan 2024 14:15:20 GMT

美股期貨盤前全下挫 台指期力守十日線
https://tw.stock.yahoo.com/news/美股期貨盤前全下挫-台指期力守十日線-141008724.html
Tue, 02 Jan 2024 14:10:08 GMT



In [21]:
import requests
from urllib.parse import unquote
import xml.etree.ElementTree as xmltree
import pyodbc

# 定義連接參數
server = '140.138.148.87'
database = '113winter_camp'
username = 'sa'
password = '1507a'

# 建立資料庫連接
conn = pyodbc.connect(f'DRIVER=SQL Server;SERVER={server};DATABASE={database};UID={username};PWD={password}')

# 創建游標
cursor = conn.cursor()

try:
    # 創建資料表(abo.s1121703)
    cursor.execute('''
        CREATE TABLE s1121703new(
            title VARCHAR(MAX),
            link VARCHAR(MAX),
            pubDate VARCHAR(MAX)
        )
    ''')

    # 發送 HTTP 請求獲取網頁內容
    link = "https://tw.stock.yahoo.com/rss?category=intl-markets"
    response = requests.get(link)

    # 從 XML 字串建立 ElementTree
    element = xmltree.fromstring(response.text)  # 從XML字串建立 Element Tree

    # 初始化各個資料的列表
    title = []
    link = []
    pubDate = []

    interests = ['title', 'link', 'pubDate']

    # 處理 XML 回應中的所有項目
    for item in element.findall('./channel/item')[0:5]:
        # 初始化變數
        item_title = None
        item_link = None
        item_pubDate = None

        for b in item:
            # 檢查元素是否在 interests 的列表中並且有值
            if b.tag == 'title' and b.text:
                item_title = unquote(b.text)
            elif b.tag == 'link' and b.text:
                item_link = unquote(b.text)
            elif b.tag == 'pubDate' and b.text:
                item_pubDate = unquote(b.text)

        # 檢查所有必需元素是否存在
        if all([item_title, item_link, item_pubDate]):
            title.append(item_title)
            link.append(item_link)
            pubDate.append(item_pubDate)

    # 將資料匯入資料庫
    for i in range(len(title)):
        cursor.execute(f"""
            INSERT INTO s1121703new(title, link, pubDate) 
            VALUES('{title[i]}', '{link[i]}', '{pubDate[i]}')
        """)

    # 從資料庫抓取資料,檢查剛才是否匯入成功
    check = f"""
        SELECT * FROM s1121703new
    """

    cursor.execute(check)

    # 取得所有檢索結果
    result = cursor.fetchall()
    print(result)

    # 提交更改
    conn.commit()

except Exception as e:
    # 如果出現錯誤,回滾更改 
    conn.rollback()
    print('出現錯誤:', e)

finally:
    # 關閉游標和連接
    cursor.close()
    conn.close()

[('李強提前透露中國GDP增長5.2% 遭專家質疑', 'https://tw.news.yahoo.com/李強提前透露中國gdp增長5-2-遭專家質疑-091834803.html', 'Wed, 17 Jan 2024 09:18:34 GMT'), ('2023出國旅遊破千萬人次 永豐ATM逾8成領日圓', 'https://tw.stock.yahoo.com/news/2023出國旅遊破千萬人次-永豐atm逾8成領日圓-093206742.html', 'Wed, 17 Jan 2024 09:32:06 GMT'), ('《國際產業》走出4年低谷 雷諾2023年銷量轉成長9%', 'https://tw.stock.yahoo.com/news/國際產業-走出4年低谷-雷諾2023年銷量轉成長9-093047521.html', 'Wed, 17 Jan 2024 09:30:47 GMT'), ('《生醫股》藥華藥新藥治療原發性骨髓纖維化 申請美FDA三期臨床', 'https://tw.stock.yahoo.com/news/生醫股-藥華藥新藥治療原發性骨髓纖維化-申請美fda三期臨床-085444745.html', 'Wed, 17 Jan 2024 08:54:44 GMT'), ('《國際政治》馬克宏擬2月訪基輔和軍援 稱歐「最大風險」仍是烏克蘭', 'https://tw.stock.yahoo.com/news/國際政治-馬克宏擬2月訪基輔和軍援-稱歐-最大風險-仍是烏克蘭-085422197.html', 'Wed, 17 Jan 2024 08:54:22 GMT')]
